### Notebook to develop and test the implementation of MTEB for sparse vectorization

In [1]:
import re 
import mteb
import torch
import gc
import numpy as np
#from src import tfidf_for_mteb
#from threadpoolctl import threadpool_limits

# autoreload to keep track of changes
%load_ext autoreload
%autoreload 1
%aimport src.tfidf_log

2024-11-25 15:37:31.735958: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-25 15:37:31.750757: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-25 15:37:31.755260: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-25 15:37:31.770491: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-25 15:37:32.844730: W tensorflow/compiler/tf2

#### Step 1: write a function to get a vocabulary list from a list of strings

In [ ]:
# using regex to find words for the vocabulary list 
# This function needs to be callable by the evaluators in sparse_mteb/mteb/evaluation
# therefore, it is copied into sparse_mteb/mteb/evaluation/evaluators/utils.py

def get_vocab(text: [str], token_pattern: str = r"(?u)\b\w\w+\b", lowercase: bool = True) -> [str]:
    """return a list of unique words ocurring in text that fulfill the specified token_pattern
    The default token_pattern is the one used in the scikit-learn TfidfVectorizer class
    """
    if lowercase:
        vocab = [word.lower() for sent in text for word in re.findall(token_pattern, sent)]
    else:
        vocab = [word for sent in text for word in re.findall(token_pattern, sent)]
    
    return list(set(vocab))


In [ ]:
test_list = ["These are SoMe words. What else?", "I've thought hard about what words to write.", "Words won't be enough!"]
get_vocab(text=test_list, lowercase=True)

#### Step 2: Try mteb tasks with tfidf

In [2]:
# define models to compare
tfidf_model = src.tfidf_log.Tfidf()
#glove_model = mteb.get_model("sentence-transformers/average_word_embeddings_glove.6B.300d")

##### Bitext Mining 

Task: translation: match sentence from set 1 to sentence from set 2 (in different language)

Implementation for TF-IDF: tfidf is obviously horrible at this. the sets could either be embedded together (which is how I implemented it for now), which means that the translation for each word would just be another matrix entry, or embedded separately, which also makes no sense because the first word in the english vocab will not align with the first word in french 

Changes implemented in which files?

In [ ]:
tasks = mteb.get_tasks(tasks=["FloresBitextMining"])
evaluation = mteb.MTEB(tasks=tasks)
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")


In [ ]:
glove_results = evaluation.run(glove_model, output_folder="../MTEB/sparse_results")

##### Classification

**Task**: train logistic regression classifier on train set embeddings, evaluate on performance for test set embeddings

**Implementation for TF-IDF:** compute vocab for embeddings for the union of train and test set. For multilingual datasets, the embeddings are computed for each language separately to reduce the size of embeddings (this shouldn't lead to conflicts because each language is evaluated separately)

**Changes implemented in which files?**
- AbsTaskClassification.py

**Problem Tasks**
- AmazonPolarityClassification (fails on cin cluster at float32 precision)
- probably also AmazonReviewsClassification

In [ ]:
import os
os.getcwd()

In [ ]:
tasks = mteb.get_tasks(tasks=['AmazonPolarityClassification'])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="text_embedding/MTEB/sparse_results")

In [ ]:
glove_results = evaluation.run(glove_model, output_folder="../MTEB/sparse_results")

##### Clustering

**Task**: k-means model on labeled embeddings for k clusters

**Implementation for TF-IDF**: MTEB only performs clustering on small subset of data (2048 samples) -> should we compute embeddings for this subset or entire data? (This is at least the case for the tasks that get evaluated with AbsTaskClusteringFast.py, maybe not those that get evaluated with AbsTaskClustering.py, but how do I differentiate between these?)

Embedding space is created for the entire corpus, not only the current subset. This is conceptually stronger, because then all clustering happens on the same subspace. Performance-wise, this doesn't change much, speed-wise it does.


**Changes implemented in which files?**
(All of these changes are currently commented out, except for AbsTaskClustering)
- AbsTask.py (this will most likely lead to conflicts!)
- AbsTaskClusteringFast.py
- AbsTaskClustering.py

**Problem Tasks**
- ArxivClusteringP2P

In [ ]:
tasks = mteb.get_tasks(tasks=["RedditClusteringP2P"])
evaluation = mteb.MTEB(tasks=tasks)

TF-IDF on ArXivHierarchicalClusteringS2S

results for computing embeddings on subset: v-measure: 0.4975645682072562

results for computing embeddings on whole data: "v_measure": 0.5006167526797055

TF-IDF on ArxivClusteringS2S

results for computing embeddings on whole data: "v_measure": 0.11543345526606984

TF-IDF on MedrivClusteringS2S

results for computing embeddings on subset: "v_measure": 0.15541653902737848

results for computing embeddings on whole data: 0.15547118002961202

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="text_embedding/MTEB/sparse_results")

##### Pair Classification

**Task**: classify a pair of texts as duplicates or paraphrases (binary choice)

**Implementation for TF-IDF**: probably bad performance because tfidf would only recognize a parashrase pair if the same words are used in both. Embedding should be done on the whole dataset, not on the single pairs.

**Changes implemented in which files?**
No changes necessary, embedding already happens for the entire dataset at once

In [ ]:
tasks = mteb.get_tasks(tasks=["TwitterSemEval2015"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### Reranking

**Task**: rank texts according to relevance to query. relevance is computed for each document-query pair. each query has a list of positive/relevant and negative/irrelevant documents

**Implementation for TF-IDF**: embed all texts & all queries together

**Changes implemented in which files?**
- RerankingEvaluator.py

**Problem Tasks**
- SciDocsRR (works for dtype=np.float32)
- MindSmallReranking (tried with dtype=np.float32 on CIN cluster, but fails: MemoryError: Unable to allocate 283. GiB for an array with shape (2658091, 28550) and data type float32 -> need svd dim reduction to work)

In [3]:
tasks = mteb.get_tasks(tasks=["SciDocsRR"])
evaluation = mteb.MTEB(tasks=tasks)

In [4]:
tfidf_results = evaluation.run(tfidf_model, 
                               output_folder="text_embedding/MTEB/sparse_results")#,
                               #encode_kwargs={"dtype": np.float32})

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Reranking

- SciDocsRR, s2s

AbsTask.load_data called
AbsTask.evaluate called in AbsTask.py
DatasetDict({
    validation: Dataset({
        features: ['query', 'positive', 'negative'],
        num_rows: 3981
    })
    test: Dataset({
        features: ['query', 'positive', 'negative'],
        num_rows: 3978
    })
})
get_vocab called!
lowercase is True
input: <class 'list'> of length 396561
input[0]: <class 'str'> of length 1
are all entries in vocab lowercase? True
uppercase entries: []
vocab of lenght 39564 starting with ['dimensional', 'rainforest', 'bartholin', 'voxelnet', 'disordered', 'ward', 'snac', 'milkweed', 'historical', 'tapped']
tfidf matrix shape(3978, 39564)


A total on 33447/118600 duplicate texts were found during encoding. Only encoding unique text and duplicating embeddings across.


dense matrix shape(3978, 39564)
tfidf matrix shape(85153, 39564)
dense matrix shape(85153, 39564)


In [ ]:
all(list(map(lambda x: x.islower(), ["A", "b"])))

In [ ]:
[word for word in ["A", "b"] if not word.islower()]

##### Retrieval

**Task**: select texts that are relevant for query

**Implementation for TF-IDF**: embed all texts & all queries together

**Changes implemented in which files?**
- AbsTaskRetrieval.py


In [ ]:
tasks = mteb.get_tasks(tasks=["SCIDOCS"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### STS (Semantic Textual Similarity)

**Task**: determine similarity between sentence pairs, with ground truth similarities given

**Implementation for TF-IDF**: embed all sentence pairs together or each pair on its own? Maybe test both?

**Changes implemented in which files?**
- STSEvaluator.py

In [ ]:
tasks = mteb.get_tasks(tasks=["STSBenchmark"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")

##### Summarization

**Task**: Machine-generated summaries are provided with human-generated quality scores. Task is to predict the summary quality by embedding the machine-generated summaries and corresponding human-generated summaries, and for each summary use the minimal distance to one of the human-generated summaries as a score.

**Implementation for TF-IDF**: 

**Changes implemented in which files?**


In [ ]:
tasks = mteb.get_tasks(tasks=["SummEvalSummarization.v2"])
evaluation = mteb.MTEB(tasks=tasks)

In [ ]:
tfidf_results = evaluation.run(tfidf_model, output_folder="../MTEB/sparse_results")